# Simulation-Derived 5G/UAS Cyber-Incident Telemetry

Gazebo + ROS 2 + PX4 SITL + OMNeT++/Simu5G

This notebook follows the supplied experiment document in operational order. It
contains only the required production, validation, and comparison checkpoints.


## Step 1 — Environment and experiment folders

The project uses `UAV_5GSim` as the laboratory root. The document folders are
stored under `experiments/`; the schema, validator, bridge, and synthetic
reference remain at the laboratory root. This is a path mapping only—the
required folder roles are unchanged.


In [1]:
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd().resolve()
assert HERE.name == 'UAV_5GSim', f'Run this notebook from UAV_5GSim, not {HERE}'
PROJECT = HERE.parent
EXP = HERE / 'experiments'
PATHS = {
    'data': EXP / 'data', 'data_ref': EXP / 'data_ref',
    'logs': EXP / 'logs', 'manifests': EXP / 'manifests',
    'results': EXP / 'results', 'trace': EXP / 'trace',
    'reports': EXP / 'reports', 'bridge': HERE / 'sim/bridge',
}
for path in PATHS.values(): path.mkdir(parents=True, exist_ok=True)

SCHEMA = HERE / 'schema_reference.json'
VALIDATOR = HERE / 'validate_dataset.py'
SYNTHETIC = HERE / 'data_ref/drone_network_telemetry_dataset.csv'
CANONICAL = EXP / 'data/drone_network_telemetry_cosim.csv'
COMPLETE_ALIAS = EXP / 'data/drone_network_telemetry_cosim_document_complete.csv'
SIMU5G = PROJECT / 'UGV_UAV_5G_CoSimulation/sim/simu5g/Simu5G'
OMNETPP = Path('/home/basudeo/omnetpp-6.0.1')
PX4 = PROJECT / 'UGV_UAV/px4_runtime/bin/px4'

checks = {
    'Ubuntu 24.04': '24.04' in Path('/etc/os-release').read_text(),
    'ROS 2 Jazzy': Path('/opt/ros/jazzy').is_dir(),
    'PX4 SITL': PX4.is_file(), 'OMNeT++': OMNETPP.is_dir(),
    'Simu5G': SIMU5G.is_dir(), 'schema': SCHEMA.is_file(),
    'validator': VALIDATOR.is_file(), 'synthetic reference': SYNTHETIC.is_file(),
}
for name, passed in checks.items(): print('[OK]' if passed else '[MISSING]', name)
assert all(checks.values())
print(f"[STORAGE] {shutil.disk_usage(HERE).free / 1024**3:.1f} GiB free")


[OK] Ubuntu 24.04
[OK] ROS 2 Jazzy
[OK] PX4 SITL
[OK] OMNeT++
[OK] Simu5G
[OK] schema
[OK] validator
[OK] synthetic reference
[STORAGE] 172.9 GiB free


## Step 2 — Required 44-column schema and metric contract

Each output row represents one communication flow in one aggregation window.
Raw traffic measurements come from Simu5G/INET; rates, flags, and cumulative
values are derived by the bridge. Label fields are never used to derive input
features.


In [2]:
required_columns = [
 'timestamp','organization_id','fleet_id','drone_id','mission_id','domain','mission_type',
 'source_component','destination_component','source_ip','destination_ip','source_port',
 'destination_port','protocol','network_slice_id','cell_id','session_id','packet_count',
 'byte_count','packets_per_second','throughput_kbps','latency_ms','jitter_ms',
 'packet_loss_rate','retransmission_rate','connection_duration_sec','authentication_status',
 'handover_event','handover_count','signal_quality_dbm','command_channel_activity',
 'video_stream_activity','telemetry_stream_activity','api_request_count',
 'failed_connection_attempts','unusual_destination_flag','session_reuse_flag',
 'traffic_burst_flag','anomaly_score','attack_type','attack_stage','severity',
 'recommended_response','incident_label'
]
schema = json.loads(SCHEMA.read_text())
assert schema['columns'] == required_columns and len(required_columns) == 44
print('[SCHEMA PASSED] Exact 44-column order')


[SCHEMA PASSED] Exact 44-column order


## Step 3 — Scenario manifests and controlled run matrix

Every run has one YAML manifest containing mission parameters, network
parameters, expected effects, attack configuration, and time-local label
windows. The fast mobility trace is recorded as 8 m/s.


In [3]:
manifest_paths = sorted(PATHS['manifests'].glob('*.yaml'))
manifests = {path.stem: yaml.safe_load(path.read_text()) for path in manifest_paths}
required_top = {'organization_id','fleet_id','drone_id','mission_id','domain','mission_type',
                'scenario_id','scenario_family','simulation','attack','label_windows',
                'allowed_destinations','expected_effects'}
required_sim = {'drone_count','mission_speed_mps','duration_sec','aggregation_window_sec',
                'cells','background_traffic','network_slice_profile'}
for run_id, manifest in manifests.items():
    assert required_top.issubset(manifest), run_id
    assert required_sim.issubset(manifest['simulation']), run_id
    assert manifest['label_windows'], run_id
assert manifests['B_SPEED_FAST_001']['simulation']['mission_speed_mps'] == 8.0
print('[MANIFESTS PASSED]', len(manifests), 'runs')
print(pd.Series([m['scenario_family'] for m in manifests.values()]).value_counts().sort_index())


[MANIFESTS PASSED] 46 runs
benign                   9
benign_duration          3
benign_sensitivity       9
benign_slice_profile     2
malicious               15
suspicious               8
Name: count, dtype: int64


## Step 4 — Gazebo/PX4 mission and ROS 2 recording evidence

The completed experiment contains Gazebo, PX4, MAVROS, and ROS 2 bag evidence
for every run. Controlled network comparisons intentionally reuse measured
flight recordings so that only the selected network condition changes. Reuse is
recorded with `physical_source.json`; it is not presented as a new physical
flight.


In [4]:
bag_inodes = {}
for run_id in manifests:
    run = PATHS['logs'] / run_id
    required = [run/'gazebo.log', run/'px4.log', run/'mavros.log',
                run/'rosbag/metadata.yaml', run/'ros_pose_trace.csv',
                run/'topic_measurements.csv']
    assert all(path.is_file() and path.stat().st_size > 0 for path in required), run_id
    bags = list((run/'rosbag').glob('*.mcap')); assert bags, run_id
    stat = bags[0].stat(); bag_inodes.setdefault((stat.st_dev, stat.st_ino), []).append(run_id)
print('[ROS/PX4 EVIDENCE PASSED]', len(manifests), 'run folders')
print('[UNIQUE MEASURED FLIGHTS]', len(bag_inodes))
print('[POLICY] Trace reuse isolates network variables and is recorded per run')


[ROS/PX4 EVIDENCE PASSED] 46 run folders
[UNIQUE MEASURED FLIGHTS] 5
[POLICY] Trace reuse isolates network variables and is recorded per run


## Step 5 — Pose export and Simu5G mobility

Each run contains the exported Cartesian pose trace and the Simu5G-compatible
mobility trace. The trace is the shared-clock link between vehicle motion and
network simulation.


In [5]:
for run_id in manifests:
    run = PATHS['logs'] / run_id
    for path in (run/'ros_pose_trace.csv', run/'simu5g_mobility.txt'):
        assert path.is_file() and path.stat().st_size > 0, path
fast = pd.read_csv(PATHS['trace']/'B_SPEED_FAST_001_mission_pose.txt', sep=r'\s+')
velocity = ((fast.x_east_m.diff()**2 + fast.y_north_m.diff()**2 + fast.z_up_m.diff()**2)**0.5) / fast.time_sec.diff()
assert 7.5 <= velocity[velocity > .05].median() <= 8.5
print('[POSE EVIDENCE PASSED]', len(manifests), 'runs')
print(f'[FAST TRACE] median moving speed={velocity[velocity>.05].median():.2f} m/s')


[POSE EVIDENCE PASSED] 46 runs
[FAST TRACE] median moving speed=7.98 m/s


## Step 6 — Simu5G network outputs

Every scenario has scalar/vector evidence and an exported network-metrics CSV.
Large completed raw files may be gzip-compressed; compression does not alter
their content or provenance. Network-slice profiles represent separate logical
mission/video/telemetry/backend flows. Native scheduler-enforced slicing is not
claimed because it is unavailable in the installed Simu5G source.


In [6]:
def one_exists(folder, names): return any((folder/name).is_file() for name in names)
for run_id in manifests:
    raw = PATHS['logs']/run_id/'network/raw'
    assert one_exists(raw, ['baseline.sca','pilot.sca']), run_id
    assert one_exists(raw, ['baseline.vec','baseline.vec.gz','pilot.vec','pilot.vec.gz']), run_id
    assert one_exists(raw, ['network_metrics.csv','network_metrics.csv.gz']), run_id
profiles = {m['simulation']['network_slice_profile'] for m in manifests.values()}
assert {'single_mission_slice','mission_and_video','mission_telemetry_backend'}.issubset(profiles)
print('[SIMU5G EVIDENCE PASSED]', len(manifests), 'runs')
print('[LOGICAL FLOW PROFILES]', sorted(profiles))


[SIMU5G EVIDENCE PASSED] 46 runs
[LOGICAL FLOW PROFILES] ['mission_and_video', 'mission_telemetry_backend', 'single_mission_slice']


## Step 7 — Bridge assembly and per-run validation

`assemble.py` derives one row per flow/window from the manifest, ROS evidence,
and Simu5G export. `concat.py` combines per-run files. Every per-run CSV must
pass the authoritative validator before final concatenation.


In [7]:
bridge_files = [PATHS['bridge']/'assemble.py', PATHS['bridge']/'concat.py']
compile_result = subprocess.run([sys.executable, '-m', 'py_compile', *map(str, bridge_files)], capture_output=True, text=True)
assert compile_result.returncode == 0, compile_result.stderr
validation_failures = []
for run_id in manifests:
    dataset = PATHS['logs']/run_id/'network/data'/f'{run_id}_cosim.csv'
    result = subprocess.run([sys.executable, str(VALIDATOR), str(dataset), str(SCHEMA)], capture_output=True, text=True)
    if result.returncode: validation_failures.append((run_id, result.stdout + result.stderr))
assert not validation_failures, validation_failures
print('[BRIDGE PASSED]', [path.name for path in bridge_files])
print('[PER-RUN VALIDATION PASSED]', len(manifests), '/', len(manifests))


[BRIDGE PASSED] ['assemble.py', 'concat.py']
[PER-RUN VALIDATION PASSED] 46 / 46


## Step 8 — Required scenario and parameter coverage

This checkpoint covers benign, suspicious, and six single-attack classes plus
the required speed, duration, window, load, cell, drone-count, wireless, slice,
intensity, and timing variations.


In [8]:
attacks = {m['attack']['attack_type'] for m in manifests.values()}
required_attacks = {'denial_of_service','session_hijacking','identity_theft',
                    'control_protocol_attack','lateral_movement','network_application_attack'}
assert required_attacks.issubset(attacks)
families = pd.Series([m['scenario_family'] for m in manifests.values()]).value_counts()
assert families.get('suspicious', 0) >= 8
sims = [m['simulation'] for m in manifests.values()]
assert {0.5,1.0,5.0}.issubset({float(x['aggregation_window_sec']) for x in sims})
assert {180,300,600,1200}.issubset({int(x['duration_sec']) for x in sims})
assert {1,2,3}.issubset({int(x['cells']) for x in sims})
assert {1,3,5}.issubset({int(x['drone_count']) for x in sims})
assert {'low','medium','high'}.issubset({x['background_traffic'] for x in sims})
assert {'off','low','high'}.issubset({m['attack']['intensity'] for m in manifests.values()})
print('[SCENARIO COVERAGE PASSED]')
print('attacks:', sorted(required_attacks))
print('families:', families.to_dict())


[SCENARIO COVERAGE PASSED]
attacks: ['control_protocol_attack', 'denial_of_service', 'identity_theft', 'lateral_movement', 'network_application_attack', 'session_hijacking']
families: {'malicious': 15, 'benign': 9, 'benign_sensitivity': 9, 'suspicious': 8, 'benign_duration': 3, 'benign_slice_profile': 2}


## Step 9 — Concatenate the canonical final dataset

The source-run order is taken from the preserved document-complete alias. The
bridge concatenates the same per-run validated files into the exact required
final name: `experiments/data/drone_network_telemetry_cosim.csv`.


In [9]:
source_order = pd.read_csv(COMPLETE_ALIAS, usecols=['mission_id']).mission_id.drop_duplicates().tolist()
assert len(source_order) == 37
inputs = [PATHS['logs']/run_id/'network/data'/f'{run_id}_cosim.csv' for run_id in source_order]
command = [sys.executable, str(PATHS['bridge']/'concat.py'), *map(str, inputs),
           '--out', str(CANONICAL), '--schema', str(SCHEMA)]
result = subprocess.run(command, capture_output=True, text=True)
assert result.returncode == 0, result.stdout + result.stderr
final = pd.read_csv(CANONICAL)
assert final.shape == (25490, 44) and final.mission_id.nunique() == 37
shutil.copy2(CANONICAL, COMPLETE_ALIAS)
print('[FINAL CONCATENATION PASSED]', final.shape, 'missions=', final.mission_id.nunique())


[FINAL CONCATENATION PASSED] (25490, 44) missions= 37


## Step 10 — Final schema, response mapping, provenance, and leakage audit


In [10]:
result = subprocess.run([sys.executable, str(VALIDATOR), str(CANONICAL), str(SCHEMA)], capture_output=True, text=True)
print(result.stdout, end=''); assert result.returncode == 0, result.stderr
assert list(final.columns) == required_columns
for run_id in source_order:
    audit = json.loads((PATHS['reports']/f'{run_id}_leakage_audit.json').read_text())
    assert audit['passed'], run_id
label_fields = {'attack_type','attack_stage','severity','recommended_response','incident_label'}
evaluation = json.loads((PATHS['reports']/'document_complete_model_evaluation.json').read_text())
assert not set(evaluation['train_missions']) & set(evaluation['test_missions'])
print('[LEAKAGE AUDIT PASSED]', len(source_order), 'source runs')
print('[CLASSES]', final.incident_label.value_counts().to_dict())


RESULT: PASSED (25490 rows, 44 columns)
[LEAKAGE AUDIT PASSED] 37 source runs
[CLASSES] {'normal': 21282, 'malicious': 2970, 'suspicious': 1238}


## Step 11 — Machine-learning and synthetic-baseline comparison

The preserved evaluation uses complete mission identifiers for splitting and
excludes label/response fields from features. It reports raw measurements, raw
plus 5G context, security indicators, and the complete feature set. This cell
checks the required accuracy, macro-F1, confusion matrices, feature importance,
false-alarm rate, and synthetic comparison outputs without changing the schema.


In [11]:
model_report = json.loads((PATHS['reports']/'document_complete_model_evaluation.json').read_text())
groups = model_report['feature_groups']
assert {'raw_network','raw_plus_5g_context','security_indicators','complete'} == set(groups)
for group in groups.values():
    for model in group.values():
        assert {'accuracy','macro_f1','confusion_matrix','false_alarm_rate'}.issubset(model)
assert model_report['feature_importance']
best = groups['complete']['random_forest']
synthetic = model_report['synthetic_reference_best']
distribution = pd.read_csv(PATHS['reports']/'document_complete_vs_synthetic_distributions.csv')
assert not distribution.empty
print(f"[MEASURED] accuracy={best['accuracy']:.4f}, macro-F1={best['macro_f1']:.4f}, false-alarm={best['false_alarm_rate']:.4f}")
print(f"[SYNTHETIC] accuracy={synthetic['accuracy']:.4f}, macro-F1={synthetic['macro_f1']:.4f}")
print('[CONFUSION MATRIX]', best['confusion_matrix'])
print('[TOP FEATURES]', model_report['feature_importance'][:10])


[MEASURED] accuracy=0.9278, macro-F1=0.7384, false-alarm=0.0122
[SYNTHETIC] accuracy=0.9988, macro-F1=0.9986
[CONFUSION MATRIX] [[12995, 96, 65], [386, 259, 39], [468, 69, 1173]]
[TOP FEATURES] [{'feature': 'numeric__connection_duration_sec', 'importance': 0.38976449748223324}, {'feature': 'numeric__signal_quality_dbm', 'importance': 0.2083734107578984}, {'feature': 'numeric__handover_event', 'importance': 0.0840610417989947}, {'feature': 'numeric__anomaly_score', 'importance': 0.0399846152912656}, {'feature': 'numeric__latency_ms', 'importance': 0.03857384483363355}, {'feature': 'numeric__handover_count', 'importance': 0.022688626168749164}, {'feature': 'numeric__unusual_destination_flag', 'importance': 0.022486746788164452}, {'feature': 'categorical__cell_id_gnb_1', 'importance': 0.02087562872122748}, {'feature': 'categorical__cell_id_gnb_2', 'importance': 0.019401684177616008}, {'feature': 'numeric__packets_per_second', 'importance': 0.019247545648769505}]


## Step 12 — Final deliverables gate

This final cell checks only the deliverables listed in the supplied experiment
document.


In [12]:
deliverables = {
 'scenario manifests': len(manifest_paths) == 46,
 'ROS 2 logs': all((PATHS['logs']/run_id/'rosbag/metadata.yaml').is_file() for run_id in manifests),
 'pose traces': all((PATHS['logs']/run_id/'simu5g_mobility.txt').is_file() for run_id in manifests),
 'Simu5G outputs': all((PATHS['logs']/run_id/'network').is_dir() for run_id in manifests),
 'bridge scripts': all(path.is_file() for path in bridge_files),
 'canonical final dataset': CANONICAL.is_file() and final.shape == (25490,44),
 'validation report': (PATHS['reports']/'document_complete_validation.txt').is_file(),
 'ML comparison report': (PATHS['reports']/'document_complete_model_evaluation.json').is_file(),
}
for name, passed in deliverables.items(): print('[PASSED]' if passed else '[FAILED]', name)
assert all(deliverables.values())
summary = {
 'status':'passed', 'rows':len(final), 'columns':len(final.columns),
 'missions':final.mission_id.nunique(), 'runs_with_evidence':len(manifests),
 'unique_measured_flights':len(bag_inodes),
 'canonical_dataset':str(CANONICAL.relative_to(HERE)),
 'trace_reuse_policy':'controlled network comparisons reuse recorded flight evidence with explicit provenance',
 'slice_scope':'logical application-flow profiles; native scheduler isolation is not claimed',
}
(PATHS['reports']/'final_dataset_manifest.json').write_text(json.dumps({
 'dataset':'experiments/data/drone_network_telemetry_cosim.csv',
 'rows':len(final),'columns':len(final.columns),'missions':final.mission_id.nunique(),
 'sha256':hashlib.sha256(CANONICAL.read_bytes()).hexdigest(),
 'class_distribution':final.incident_label.value_counts().to_dict(),
 'source_runs':source_order,'validation':'passed',
 'core_dataset_preserved':'experiments/data/drone_network_telemetry_cosim_core.csv',
 'document_complete_alias':'experiments/data/drone_network_telemetry_cosim_document_complete.csv'
},indent=2)+'\n')
(PATHS['reports']/'document_only_notebook_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print('[DOCUMENT EXPERIMENT COMPLETE]', summary)


[PASSED] scenario manifests
[PASSED] ROS 2 logs
[PASSED] pose traces
[PASSED] Simu5G outputs
[PASSED] bridge scripts
[PASSED] canonical final dataset
[PASSED] validation report
[PASSED] ML comparison report
[DOCUMENT EXPERIMENT COMPLETE] {'status': 'passed', 'rows': 25490, 'columns': 44, 'missions': 37, 'runs_with_evidence': 46, 'unique_measured_flights': 5, 'canonical_dataset': 'experiments/data/drone_network_telemetry_cosim.csv', 'trace_reuse_policy': 'controlled network comparisons reuse recorded flight evidence with explicit provenance', 'slice_scope': 'logical application-flow profiles; native scheduler isolation is not claimed'}


## Step 13 — Automated live headless duration validation

This section records the literal live 5-, 10-, and 20-minute tests requested
by the experiment document. Each independent run started a fresh headless
Gazebo/PX4 instance, recorded MAVROS ROS 2 topics, flew `spawn -> waypoint_1
-> waypoint_2 -> goal`, held until the requested duration, landed and
disarmed at the goal, processed the measured ROS pose through Simu5G, and
validated the resulting 44-column CSV. These tests use PX4's plain x500
because three-dimensional mapping is not part of the duration variable. This
keeps the test focused on flight and communication endurance.

To deliberately repeat all three unattended tests from a terminal:

```bash
cd /home/basudeo/Documents/EiraX/UAV_5GSim
conda activate eirax
./automate_live_duration_tests.py
```

The cells below verify preserved measurements and do not automatically refly
35 minutes whenever the notebook is reopened.


In [13]:
LIVE_ROOT = HERE / 'experiments/live_duration_validation'
LIVE_SUMMARY_PATH = LIVE_ROOT / 'live_duration_test_summary.json'
assert LIVE_SUMMARY_PATH.is_file(), 'Run ./automate_live_duration_tests.py first'
live_summary = json.loads(LIVE_SUMMARY_PATH.read_text())
assert live_summary['all_passed'] is True
assert {int(x['requested_duration_sec']) for x in live_summary['tests']} == {300,600,1200}

def verify_live_duration(minutes):
    duration = minutes * 60
    test = next(x for x in live_summary['tests'] if int(x['requested_duration_sec']) == duration)
    run = HERE / test['run_directory']
    flight = test['flight']; network = test['network']
    pose = test['ros_topics']['/mavros/local_position/pose']
    assert test['status'] == 'passed'
    assert float(flight['active_duration_sec']) >= duration
    assert flight['reached'] == ['takeoff','waypoint_1','waypoint_2','goal']
    assert flight['landed_and_disarmed'] is True
    assert float(flight['final_goal_horizontal_error_m']) <= 1.0
    assert int(pose['message_count']) > 0 and float(pose['duration_sec']) >= duration
    assert network['simu5g_end_confirmed'] is True
    dataset = HERE / network['dataset']
    validation = subprocess.run([sys.executable,str(VALIDATOR),str(dataset),str(SCHEMA)],capture_output=True,text=True)
    assert validation.returncode == 0, validation.stdout + validation.stderr
    simu5g_log = (run/'simu5g.log').read_text(errors='replace')
    assert f'T={duration}' in simu5g_log and 'End.' in simu5g_log
    px4_log = (run/'px4.log').read_text(errors='replace')
    forbidden = ['Accel #0 fail','Battery unhealthy','Flight termination active']
    assert not any(message in px4_log for message in forbidden)
    return {'duration_min':minutes,'active_sec':float(flight['active_duration_sec']),
            'goal_error_m':float(flight['final_goal_horizontal_error_m']),
            'pose_messages':int(pose['message_count']),
            'ros_duration_sec':float(pose['duration_sec']),
            'simu5g_duration_sec':int(network['simulated_duration_sec']),
            'dataset_rows':int(network['dataset_rows']),
            'dataset_columns':int(network['dataset_columns']),
            'landed':bool(flight['landed_and_disarmed']),'status':test['status']}

print('[LIVE TEST EVIDENCE LOADED]', LIVE_SUMMARY_PATH)


[LIVE TEST EVIDENCE LOADED] /home/basudeo/Documents/EiraX/UAV_5GSim/experiments/live_duration_validation/live_duration_test_summary.json


### Step 13.1 — Five-minute live test


In [14]:
live_5 = verify_live_duration(5)
print('[LIVE 5-MINUTE PASSED]', live_5)


[LIVE 5-MINUTE PASSED] {'duration_min': 5, 'active_sec': 300.0047625219995, 'goal_error_m': 0.030353382182553325, 'pose_messages': 16869, 'ros_duration_sec': 337.731104102, 'simu5g_duration_sec': 300, 'dataset_rows': 600, 'dataset_columns': 44, 'landed': True, 'status': 'passed'}


### Step 13.2 — Ten-minute live test


In [15]:
live_10 = verify_live_duration(10)
print('[LIVE 10-MINUTE PASSED]', live_10)


[LIVE 10-MINUTE PASSED] {'duration_min': 10, 'active_sec': 600.0033533880014, 'goal_error_m': 0.07189689675948104, 'pose_messages': 31889, 'ros_duration_sec': 638.277914915, 'simu5g_duration_sec': 600, 'dataset_rows': 1200, 'dataset_columns': 44, 'landed': True, 'status': 'passed'}


### Step 13.3 — Twenty-minute live test


In [16]:
live_20 = verify_live_duration(20)
print('[LIVE 20-MINUTE PASSED]', live_20)


[LIVE 20-MINUTE PASSED] {'duration_min': 20, 'active_sec': 1200.0112831530023, 'goal_error_m': 0.05387052637070794, 'pose_messages': 61840, 'ros_duration_sec': 1238.492260735, 'simu5g_duration_sec': 1200, 'dataset_rows': 2400, 'dataset_columns': 44, 'landed': True, 'status': 'passed'}


### Step 13.4 — Combined live-duration completion gate

These three runs are additional literal live evidence. They do not replace or
silently modify the canonical 46-scenario cyber-telemetry dataset.


In [17]:
live_results = pd.DataFrame([live_5,live_10,live_20])
assert live_results.status.eq('passed').all()
assert live_results.landed.all()
assert live_results.dataset_columns.eq(44).all()
assert live_results.simu5g_duration_sec.tolist() == [300,600,1200]
assert live_summary['canonical_dataset_unchanged'] is True
display(live_results)
print('[LIVE DURATION REQUIREMENT COMPLETE] 5, 10 and 20 minutes passed end-to-end')
print(f"[STORAGE] {live_summary['free_gib']:.1f} GiB free after compressed evidence")


,duration_min,active_sec,goal_error_m,pose_messages,ros_duration_sec,simu5g_duration_sec,dataset_rows,dataset_columns,landed,status
0,5,300.004763,0.030353,16869,337.731104,300,600,44,True,passed
1,10,600.003353,0.071897,31889,638.277915,600,1200,44,True,passed
2,20,1200.011283,0.053871,61840,1238.492261,1200,2400,44,True,passed


[LIVE DURATION REQUIREMENT COMPLETE] 5, 10 and 20 minutes passed end-to-end
[STORAGE] 172.9 GiB free after compressed evidence
